# EDA

## import

In [1]:
import pandas as pd

## Load

In [2]:
raw_df = pd.read_csv("../data/raw_data.csv")

In [3]:
raw_df.shape

(950, 36)

In [4]:
raw_df.describe(include="all").transpose()

,count,unique,top,freq
name,272,41,2号棟,60
価格_,292,98,3680万円,14
間取り_,292,11,3LDK,104
土地面積_,292,190,100.1m2,8
建物面積_,292,175,96.88m2,8
URL,950,789,https://suumo.jp/ikkodate/saitama/sc_kawagoe/n...,12
販売スケジュール,950,33,-,763
イベント情報,950,241,-,280
所在地,950,242,埼玉県川越市大字砂,35
交通,950,618,東武東上線「和光市」歩8分/東京メトロ有楽町線「和光市」歩8分/東京メトロ副都心線「和光市」...,13


## Preprocessing

### URL

In [5]:
df = raw_df["URL"].to_frame()

### ID

In [6]:
df["ID"] = raw_df["URL"].map(lambda x: x.split("/")[-2])

In [7]:
df["ID"].describe()

count             950
unique            789
top       nc_70278963
freq               12
Name: ID, dtype: object

### category

In [8]:
df["category"] = raw_df["URL"].map(lambda x: x.split("/")[3])

In [9]:
df["category"].describe()

count          950
unique           1
top       ikkodate
freq           950
Name: category, dtype: object

### value

In [11]:
df["values"] = raw_df["価格_"].str.extract("(\d*)万円").astype("float")

In [12]:
df["values"].describe()

count     292.000000
mean     4553.934932
std       884.828750
min      3090.000000
25%      3880.000000
50%      4385.000000
75%      4980.000000
max      7760.000000
Name: values, dtype: float64

In [13]:
values = raw_df["価格"].str.extractall("(\d*)万円").unstack().droplevel(level=0, axis=1).astype("float")

In [14]:
values.describe()

match,0,1,2,3
count,950.000000,314.000000,2.000000,2.000000
mean,4227.209474,4762.232484,3944.000000,4394.000000
std,897.665403,796.965118,76.367532,288.499567
min,1980.000000,3350.000000,3890.000000,4190.000000
25%,3580.000000,4180.000000,3917.000000,4292.000000
50%,3990.000000,4580.000000,3944.000000,4394.000000
75%,4590.000000,5080.000000,3971.000000,4496.000000
max,7760.000000,7680.000000,3998.000000,4598.000000


In [15]:
df["values"].fillna(values.mean(axis=1), inplace=True)

In [16]:
df["values"].describe()

count     950.000000
mean     4309.000000
std       920.851761
min      1980.000000
25%      3680.000000
50%      4180.000000
75%      4696.000000
max      7760.000000
Name: values, dtype: float64

### site_area

In [17]:
df["site_area"] = raw_df["土地面積_"].str.extract("(\d*|\d*\.\d*)m2").astype("float")

In [18]:
df["site_area"].describe()

count    292.000000
mean      97.665788
std       23.143435
min       46.100000
25%       81.325000
50%      101.080000
75%      110.092500
max      184.170000
Name: site_area, dtype: float64

In [19]:
site_area = raw_df["土地面積"].str.extractall("(\d*|\d*\.\d*)m2").unstack().droplevel(level=0, axis=1).astype("float")

In [20]:
site_area

match,0,1
0,63.32,81.58
1,63.32,81.58
2,63.32,81.58
3,101.27,109.54
4,116.04,141.63
...,...,...
945,185.37,NaN
946,200.18,NaN
947,185.37,NaN
948,185.37,NaN


In [21]:
site_area.describe()

match,0,1
count,950.000000,376.000000
mean,92.118558,102.634574
std,25.292864,29.985908
min,38.180000,16.000000
25%,71.970000,79.407500
50%,96.000000,105.755000
75%,107.360000,120.630000
max,219.080000,184.170000


In [22]:
df["site_area"].fillna(site_area.mean(axis=1), inplace=True)

In [23]:
df["site_area"].describe()

count    950.000000
mean      93.288553
std       25.454818
min       38.180000
25%       72.220000
50%       96.500000
75%      109.510000
max      200.180000
Name: site_area, dtype: float64

### total_floor_area

In [24]:
df["total_floor_area"] = raw_df["建物面積_"].str.extract("(\d*|\d*\.\d*)m2").astype("float")

In [25]:
df["total_floor_area"].describe()

count    292.000000
mean      97.696712
std       10.986985
min       68.040000
25%       92.150000
50%       98.380000
75%      103.502500
max      130.410000
Name: total_floor_area, dtype: float64

In [26]:
total_floor_area = (
    raw_df["建物面積"]
    .str.split("、", expand=True)[0]
    .str.extractall("(\d*|\d*\.\d*)m2")
    .unstack()
    .droplevel(level=0, axis=1)
    .astype("float")
)

In [27]:
total_floor_area.describe()

match,0,1
count,950.000000,357.000000
mean,95.129768,101.449300
std,13.569759,11.140581
min,42.000000,78.040000
25%,86.955000,94.190000
50%,96.880000,100.820000
75%,102.970000,111.220000
max,158.980000,130.410000


In [28]:
df["total_floor_area"].fillna(total_floor_area.mean(axis=1), inplace=True)

In [29]:
df["total_floor_area"].describe()

count    950.000000
mean      96.634042
std       13.383694
min       42.000000
25%       89.550000
50%       98.120000
75%      104.910000
max      158.980000
Name: total_floor_area, dtype: float64

In [41]:
df.groupby(["values", "site_area", "total_floor_area"]).size().to_frame("count")

count
values site_area total_floor_area       
1980.0 52.990    42.57                 6
       53.270    42.00                 1
2380.0 61.020    58.72                 1
2480.0 40.660    69.12                 1
2580.0 43.160    75.33                 1
...                                  ...
6980.0 91.290    101.43                1
7080.0 106.010   97.29                 1
7230.0 135.010   105.16                1
7240.0 123.235   98.21                 1
7760.0 176.450   119.42                1

[558 rows x 1 columns]